# 02. Salmon quantification and gene count matrix

このノートブックは、FASTQ readを発現量テーブルへ変換する段階である。

**このノートブックの役割**

- Salmonで各サンプルのFASTQをGENCODE transcript indexに対応づける。
- transcript単位の `quant.sf` から、DESeq2に渡すgene単位のcount matrixを作る。
- `tx2gene.tsv` を使って、transcript IDとgene ID/gene nameの対応を明示する。

**入力**

- `metadata/samples.tsv`（各サンプルの `sample_id`, 群情報, R1/R2 FASTQパスを持つ表）
- FASTQファイル（`samples.tsv` の `fastq_1`, `fastq_2` が指すreadファイル）
- `reference/gencode_grch38/salmon_index/`（Salmon用のGENCODE transcript index）
- `reference/gencode_grch38/tx2gene.tsv`（transcript IDをgene単位へまとめる対応表）

**出力**

- `results/quant/salmon/<sample_id>/quant.sf`（sampleごとのtranscript定量結果。主に `NumReads` と `TPM` を使う）
- `results/counts/gene_counts.tsv`（行=gene、列=sampleのread count matrix）

**次に進む条件**

- 9サンプルすべてに `quant.sf` がある。
- gene count matrix の列名が `samples.tsv` の `sample_id` と一致している。


### `salmon quant` コマンドライン引数の説明

| 引数 | 意味 |
|------|------|
| `-i DIR` | `salmon index` で作成したindex directoryを指定する。ここではGENCODE transcript indexである。 |
| `-l A` | Library type（ライブラリ形式）の指定。`A` はAutomatic（自動判定）であり、strandnessが不明な場合に扱いやすい。 |
| `-1 FILE` | ペアエンドのR1（mate 1）FASTQファイルパスである。 |
| `-2 FILE` | ペアエンドのR2（mate 2）FASTQファイルパスである。 |
| `-p N` | 並列スレッド数。定量に使うCPU数である。 |
| `--validateMappings` | mappingの検証を行うフラグ。alignment-based定量に近い精度が得られる。 |
| `-o DIR` | 出力ディレクトリ（`results/quant/salmon/<sample_id>/`）を指定する。 |

Salmonは**quasi-mapping**（擬似mapping）によりFASTQ readをTranscriptに対応づけ、各Transcriptの発現量を**TPM（Transcripts Per Million）**および**NumReads**（推定read数）として出力する。
主な出力ファイル `quant.sf` の列は次の通りである。

| 列名 | 意味 |
|------|------|
| `Name` | Transcript ID |
| `Length` | Transcript全長（bp）|
| `EffectiveLength` | 読まれ得る有効長 |
| `TPM` | 長さ・sequencing深度補正後の発現量 |
| `NumReads` | Transcriptへの推定read数（DESeq2に渡す値）|


In [1]:
from pathlib import Path
import json
import shutil
import subprocess

import pandas as pd

PROJECT_DIR = Path("/Users/yusuke_tateishi/Documents/RNA_seq").resolve()
CONFIG = json.loads((PROJECT_DIR / "config" / "analysis_config.json").read_text(encoding="utf-8"))
SAMPLES = pd.read_csv(PROJECT_DIR / CONFIG["samples_path"], sep="\t")

QUANT_DIR = PROJECT_DIR / "results" / "quant" / "salmon"
COUNTS_DIR = PROJECT_DIR / "results" / "counts"
QUANT_DIR.mkdir(parents=True, exist_ok=True)
COUNTS_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES[["sample_id", "condition", "fastq_1", "fastq_2"]]


,sample_id,condition,fastq_1,fastq_2
0,NAC_S2_2h_1,NAC_S2_2h,raw_data/NAC_S2_2h_1/NAC_S2_2h_1_1.fq.gz,raw_data/NAC_S2_2h_1/NAC_S2_2h_1_2.fq.gz
1,NAC_S2_2h_2,NAC_S2_2h,raw_data/NAC_S2_2h_2/NAC_S2_2h_2_1.fq.gz,raw_data/NAC_S2_2h_2/NAC_S2_2h_2_2.fq.gz
2,NAC_S2_2h_3,NAC_S2_2h,raw_data/NAC_S2_2h_3/NAC_S2_2h_3_1.fq.gz,raw_data/NAC_S2_2h_3/NAC_S2_2h_3_2.fq.gz
3,Non_1,Non,raw_data/Non_1/Non_1_1.fq.gz,raw_data/Non_1/Non_1_2.fq.gz
4,Non_2,Non,raw_data/Non_2/Non_2_1.fq.gz,raw_data/Non_2/Non_2_2.fq.gz
5,Non_3,Non,raw_data/Non_3/Non_3_1.fq.gz,raw_data/Non_3/Non_3_2.fq.gz
6,Oxi_2h_1,Oxi_2h,raw_data/Oxi_2h_1/Oxi_2h_1_1.fq.gz,raw_data/Oxi_2h_1/Oxi_2h_1_2.fq.gz
7,Oxi_2h_2,Oxi_2h,raw_data/Oxi_2h_2/Oxi_2h_2_1.fq.gz,raw_data/Oxi_2h_2/Oxi_2h_2_2.fq.gz
8,Oxi_2h_3,Oxi_2h,raw_data/Oxi_2h_3/Oxi_2h_3_1.fq.gz,raw_data/Oxi_2h_3/Oxi_2h_3_2.fq.gz


## 参照ファイルが準備済みか確認する

ここが `False` なら、先に `00b_reference_setup_gencode_grch38.ipynb` を実行する。


In [2]:
def resolve_project_path(path_text: str) -> Path:
    path = Path(path_text)
    return path if path.is_absolute() else PROJECT_DIR / path

reference = CONFIG["reference"]
reference_paths = {
    "salmon_index": resolve_project_path(reference["salmon_index"]),
    "tx2gene_path": resolve_project_path(reference["tx2gene_path"]),
    "transcript_fasta": resolve_project_path(reference.get("transcript_fasta", "")) if reference.get("transcript_fasta") else None,
    "gtf_path": resolve_project_path(reference["gtf_path"]) if reference.get("gtf_path") else None,
}

readiness = {
    name: (path.exists() if path is not None else False)
    for name, path in reference_paths.items()
}
readiness


{'salmon_index': True,
 'tx2gene_path': True,
 'transcript_fasta': True,
 'gtf_path': True}

## Salmonコマンドを作る

`--libType A` は、Salmonにライブラリの向きを自動推定させる設定である。初心者の最初の解析ではこの設定が扱いやすい。


In [3]:
THREADS = int(CONFIG["quantification"].get("threads", 8))
LIBRARY_TYPE = CONFIG["quantification"].get("salmon", {}).get("library_type", "A")

salmon_commands = []
for _, sample in SAMPLES.iterrows():
    output_dir = QUANT_DIR / sample["sample_id"]
    command = [
        "salmon",
        "quant",
        "-i",
        str(reference_paths["salmon_index"]),
        "-l",
        LIBRARY_TYPE,
        "-1",
        str(PROJECT_DIR / sample["fastq_1"]),
        "-2",
        str(PROJECT_DIR / sample["fastq_2"]),
        "-p",
        str(THREADS),
        "--validateMappings",
        "-o",
        str(output_dir),
    ]
    salmon_commands.append(command)

command_path = QUANT_DIR / "salmon_commands.txt"
command_path.write_text("\n\n".join(" ".join(command) for command in salmon_commands) + "\n", encoding="utf-8")
print("Commands written to:", command_path.relative_to(PROJECT_DIR))
print("First command:")
print(" ".join(salmon_commands[0]))


Commands written to: results/quant/salmon/salmon_commands.txt
First command:
salmon quant -i /Users/yusuke_tateishi/Documents/RNA_seq/reference/gencode_grch38/salmon_index -l A -1 /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/NAC_S2_2h_1/NAC_S2_2h_1_1.fq.gz -2 /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/NAC_S2_2h_1/NAC_S2_2h_1_2.fq.gz -p 8 --validateMappings -o /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/NAC_S2_2h_1


## Salmonを実行する

初回だけ `RUN_SALMON = True` にする。処理時間はデータ量とCPU数に依存する。


In [4]:
RUN_SALMON = True

if RUN_SALMON:
    salmon = shutil.which("salmon")
    if salmon is None:
        raise RuntimeError("salmon was not found. Activate the rna-seq environment first.")
    if not reference_paths["salmon_index"].exists():
        raise FileNotFoundError(reference_paths["salmon_index"])
    for command in salmon_commands:
        command[0] = salmon
        print("Running:", " ".join(command))
        subprocess.run(command, check=True)
else:
    print("Set RUN_SALMON = True after the Salmon index is ready.")


Running: /Users/yusuke_tateishi/miniconda3/envs/rna-seq/bin/salmon quant -i /Users/yusuke_tateishi/Documents/RNA_seq/reference/gencode_grch38/salmon_index -l A -1 /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/NAC_S2_2h_1/NAC_S2_2h_1_1.fq.gz -2 /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/NAC_S2_2h_1/NAC_S2_2h_1_2.fq.gz -p 8 --validateMappings -o /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/NAC_S2_2h_1


Version Server Response: Not Found
### salmon (selective-alignment-based) v1.11.4
### [ program ] => salmon 
### [ command ] => quant 
### [ index ] => { /Users/yusuke_tateishi/Documents/RNA_seq/reference/gencode_grch38/salmon_index }
### [ libType ] => { A }
### [ mates1 ] => { /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/NAC_S2_2h_1/NAC_S2_2h_1_1.fq.gz }
### [ mates2 ] => { /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/NAC_S2_2h_1/NAC_S2_2h_1_2.fq.gz }
### [ threads ] => { 8 }
### [ validateMappings ] => { }
### [ output ] => { /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/NAC_S2_2h_1 }
Logs will be written to /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/NAC_S2_2h_1/logs
[2026-05-22 14:44:53.277] [jointLog] [info] setting maxHashResizeThreads to 8
[2026-05-22 14:44:53.277] [jointLog] [info] Fragment incompatibility prior below threshold.  Incompatible fragments will be ignored.
[2026-05-22 14:44:53.277] [jointLog] [info] Usage of --validateMa

Running: /Users/yusuke_tateishi/miniconda3/envs/rna-seq/bin/salmon quant -i /Users/yusuke_tateishi/Documents/RNA_seq/reference/gencode_grch38/salmon_index -l A -1 /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/NAC_S2_2h_2/NAC_S2_2h_2_1.fq.gz -2 /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/NAC_S2_2h_2/NAC_S2_2h_2_2.fq.gz -p 8 --validateMappings -o /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/NAC_S2_2h_2


Version Server Response: Not Found
### salmon (selective-alignment-based) v1.11.4
### [ program ] => salmon 
### [ command ] => quant 
### [ index ] => { /Users/yusuke_tateishi/Documents/RNA_seq/reference/gencode_grch38/salmon_index }
### [ libType ] => { A }
### [ mates1 ] => { /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/NAC_S2_2h_2/NAC_S2_2h_2_1.fq.gz }
### [ mates2 ] => { /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/NAC_S2_2h_2/NAC_S2_2h_2_2.fq.gz }
### [ threads ] => { 8 }
### [ validateMappings ] => { }
### [ output ] => { /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/NAC_S2_2h_2 }
Logs will be written to /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/NAC_S2_2h_2/logs
[2026-05-22 14:47:49.181] [jointLog] [info] setting maxHashResizeThreads to 8
[2026-05-22 14:47:49.181] [jointLog] [info] Fragment incompatibility prior below threshold.  Incompatible fragments will be ignored.
[2026-05-22 14:47:49.181] [jointLog] [info] Usage of --validateMa

Running: /Users/yusuke_tateishi/miniconda3/envs/rna-seq/bin/salmon quant -i /Users/yusuke_tateishi/Documents/RNA_seq/reference/gencode_grch38/salmon_index -l A -1 /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/NAC_S2_2h_3/NAC_S2_2h_3_1.fq.gz -2 /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/NAC_S2_2h_3/NAC_S2_2h_3_2.fq.gz -p 8 --validateMappings -o /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/NAC_S2_2h_3


...

Version Server Response: Not Found
### salmon (selective-alignment-based) v1.11.4
### [ program ] => salmon 
### [ command ] => quant 
### [ index ] => { /Users/yusuke_tateishi/Documents/RNA_seq/reference/gencode_grch38/salmon_index }
### [ libType ] => { A }
### [ mates1 ] => { /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/Oxi_2h_1/Oxi_2h_1_1.fq.gz }
### [ mates2 ] => { /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/Oxi_2h_1/Oxi_2h_1_2.fq.gz }
### [ threads ] => { 8 }
### [ validateMappings ] => { }
### [ output ] => { /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/Oxi_2h_1 }
Logs will be written to /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/Oxi_2h_1/logs
[2026-05-22 15:00:57.851] [jointLog] [info] setting maxHashResizeThreads to 8
[2026-05-22 15:00:57.851] [jointLog] [info] Fragment incompatibility prior below threshold.  Incompatible fragments will be ignored.
[2026-05-22 15:00:57.851] [jointLog] [info] Usage of --validateMappings implies use

Running: /Users/yusuke_tateishi/miniconda3/envs/rna-seq/bin/salmon quant -i /Users/yusuke_tateishi/Documents/RNA_seq/reference/gencode_grch38/salmon_index -l A -1 /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/Oxi_2h_2/Oxi_2h_2_1.fq.gz -2 /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/Oxi_2h_2/Oxi_2h_2_2.fq.gz -p 8 --validateMappings -o /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/Oxi_2h_2


Version Server Response: Not Found
### salmon (selective-alignment-based) v1.11.4
### [ program ] => salmon 
### [ command ] => quant 
### [ index ] => { /Users/yusuke_tateishi/Documents/RNA_seq/reference/gencode_grch38/salmon_index }
### [ libType ] => { A }
### [ mates1 ] => { /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/Oxi_2h_2/Oxi_2h_2_1.fq.gz }
### [ mates2 ] => { /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/Oxi_2h_2/Oxi_2h_2_2.fq.gz }
### [ threads ] => { 8 }
### [ validateMappings ] => { }
### [ output ] => { /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/Oxi_2h_2 }
Logs will be written to /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/Oxi_2h_2/logs
[2026-05-22 15:03:32.684] [jointLog] [info] setting maxHashResizeThreads to 8
[2026-05-22 15:03:32.684] [jointLog] [info] Fragment incompatibility prior below threshold.  Incompatible fragments will be ignored.
[2026-05-22 15:03:32.684] [jointLog] [info] Usage of --validateMappings implies use

Running: /Users/yusuke_tateishi/miniconda3/envs/rna-seq/bin/salmon quant -i /Users/yusuke_tateishi/Documents/RNA_seq/reference/gencode_grch38/salmon_index -l A -1 /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/Oxi_2h_3/Oxi_2h_3_1.fq.gz -2 /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/Oxi_2h_3/Oxi_2h_3_2.fq.gz -p 8 --validateMappings -o /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/Oxi_2h_3


Version Server Response: Not Found
### salmon (selective-alignment-based) v1.11.4
### [ program ] => salmon 
### [ command ] => quant 
### [ index ] => { /Users/yusuke_tateishi/Documents/RNA_seq/reference/gencode_grch38/salmon_index }
### [ libType ] => { A }
### [ mates1 ] => { /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/Oxi_2h_3/Oxi_2h_3_1.fq.gz }
### [ mates2 ] => { /Users/yusuke_tateishi/Documents/RNA_seq/raw_data/Oxi_2h_3/Oxi_2h_3_2.fq.gz }
### [ threads ] => { 8 }
### [ validateMappings ] => { }
### [ output ] => { /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/Oxi_2h_3 }
Logs will be written to /Users/yusuke_tateishi/Documents/RNA_seq/results/quant/salmon/Oxi_2h_3/logs
[2026-05-22 15:06:36.148] [jointLog] [info] setting maxHashResizeThreads to 8
[2026-05-22 15:06:36.148] [jointLog] [info] Fragment incompatibility prior below threshold.  Incompatible fragments will be ignored.
[2026-05-22 15:06:36.148] [jointLog] [info] Usage of --validateMappings implies use

## Salmon結果がそろっているか確認する

各サンプルの `quant.sf` が存在すれば、次にgene単位へ集計できる。


In [5]:
quant_status = []
for sample_id in SAMPLES["sample_id"]:
    quant_path = QUANT_DIR / sample_id / "quant.sf"
    quant_status.append({"sample_id": sample_id, "quant_sf_exists": quant_path.exists(), "path": str(quant_path.relative_to(PROJECT_DIR))})

quant_status = pd.DataFrame(quant_status)
quant_status


,sample_id,quant_sf_exists,path
0,NAC_S2_2h_1,True,results/quant/salmon/NAC_S2_2h_1/quant.sf
1,NAC_S2_2h_2,True,results/quant/salmon/NAC_S2_2h_2/quant.sf
2,NAC_S2_2h_3,True,results/quant/salmon/NAC_S2_2h_3/quant.sf
3,Non_1,True,results/quant/salmon/Non_1/quant.sf
4,Non_2,True,results/quant/salmon/Non_2/quant.sf
5,Non_3,True,results/quant/salmon/Non_3/quant.sf
6,Oxi_2h_1,True,results/quant/salmon/Oxi_2h_1/quant.sf
7,Oxi_2h_2,True,results/quant/salmon/Oxi_2h_2/quant.sf
8,Oxi_2h_3,True,results/quant/salmon/Oxi_2h_3/quant.sf


## gene count matrixを作る

Salmonの `quant.sf` にはtranscript単位の `NumReads` が入っている。ここでは `tx2gene.tsv` を使ってgene単位に合計する。

これはDESeq2へ渡すための表である。行がgene、列がsampleになる。


In [13]:
tx2gene_path = reference_paths["tx2gene_path"]
if not tx2gene_path.exists():
    raise FileNotFoundError(tx2gene_path)

tx2gene = pd.read_csv(tx2gene_path, sep="\t")
required_columns = {"transcript_id", "transcript_name", "gene_name", "gene_id"}
if not required_columns.issubset(tx2gene.columns):
    raise ValueError(f"tx2gene.tsv must contain columns: {required_columns}")
tx2gene = tx2gene[["transcript_id", "transcript_name", "gene_name", "gene_id"]].drop_duplicates()
tx2gene.head()

,transcript_id,transcript_name,gene_name,gene_id
0,ENST00000832824.1,DDX11L16-260,DDX11L16,ENSG00000290825.2
1,ENST00000832825.1,DDX11L16-261,DDX11L16,ENSG00000290825.2
2,ENST00000832826.1,DDX11L16-262,DDX11L16,ENSG00000290825.2
3,ENST00000832827.1,DDX11L16-263,DDX11L16,ENSG00000290825.2
4,ENST00000832828.1,DDX11L16-264,DDX11L16,ENSG00000290825.2


In [14]:
BUILD_GENE_COUNTS_FROM_SALMON = True

if BUILD_GENE_COUNTS_FROM_SALMON:
    per_sample_counts = []
    for sample_id in SAMPLES["sample_id"]:
        quant_path = QUANT_DIR / sample_id / "quant.sf"
        if not quant_path.exists():
            raise FileNotFoundError(quant_path)
        quant = pd.read_csv(quant_path, sep="\t", usecols=["Name", "NumReads"]).rename(columns={"Name": "transcript_id"})
        merged = quant.merge(tx2gene, on="transcript_id", how="inner")
        gene_counts = merged.groupby("gene_id", as_index=True)["NumReads"].sum()
        gene_counts.name = sample_id
        per_sample_counts.append(gene_counts)

    count_matrix = pd.concat(per_sample_counts, axis=1).fillna(0).round().astype(int)
    count_matrix.insert(0, "gene_id", count_matrix.index)
    out_path = PROJECT_DIR / CONFIG["differential_expression"]["count_matrix_path"]
    out_path.parent.mkdir(parents=True, exist_ok=True)
    count_matrix.to_csv(out_path, sep="\t", index=False)
    print("Wrote:", out_path.relative_to(PROJECT_DIR))
    display(count_matrix.head())
else:
    print("Set BUILD_GENE_COUNTS_FROM_SALMON = True after all quant.sf files exist.")


Wrote: results/counts/gene_counts.tsv


,gene_id,NAC_S2_2h_1,NAC_S2_2h_2,NAC_S2_2h_3,Non_1,Non_2,Non_3,Oxi_2h_1,Oxi_2h_2,Oxi_2h_3
gene_id,,,,,,,,,,
ENSG00000000003.17,ENSG00000000003.17,1,0,1,1,1,3,0,0,3
ENSG00000000005.6,ENSG00000000005.6,0,0,0,0,0,0,0,0,0
ENSG00000000419.15,ENSG00000000419.15,838,720,738,753,750,609,688,833,731
ENSG00000000457.15,ENSG00000000457.15,242,232,267,204,247,214,199,222,215
ENSG00000000460.18,ENSG00000000460.18,453,341,400,369,413,318,294,313,294


## count matrixの形を確認する

count matrixが既にあれば、行数・列数・列名を確認する。


In [15]:
count_path = PROJECT_DIR / CONFIG["differential_expression"]["count_matrix_path"]
if count_path.exists():
    counts = pd.read_csv(count_path, sep="\t")
    print("shape:", counts.shape)
    print("columns:", list(counts.columns[:10]))
    display(counts.head())
    missing_samples = set(SAMPLES["sample_id"]) - set(counts.columns)
    extra_samples = set(counts.columns) - {"gene_id"} - set(SAMPLES["sample_id"])
    print("missing_samples:", sorted(missing_samples))
    print("extra_samples:", sorted(extra_samples))
else:
    print("No gene count matrix yet:", count_path.relative_to(PROJECT_DIR))


shape: (78247, 10)
columns: ['gene_id', 'NAC_S2_2h_1', 'NAC_S2_2h_2', 'NAC_S2_2h_3', 'Non_1', 'Non_2', 'Non_3', 'Oxi_2h_1', 'Oxi_2h_2', 'Oxi_2h_3']


,gene_id,NAC_S2_2h_1,NAC_S2_2h_2,NAC_S2_2h_3,Non_1,Non_2,Non_3,Oxi_2h_1,Oxi_2h_2,Oxi_2h_3
0,ENSG00000000003.17,1,0,1,1,1,3,0,0,3
1,ENSG00000000005.6,0,0,0,0,0,0,0,0,0
2,ENSG00000000419.15,838,720,738,753,750,609,688,833,731
3,ENSG00000000457.15,242,232,267,204,247,214,199,222,215
4,ENSG00000000460.18,453,341,400,369,413,318,294,313,294


missing_samples: []
extra_samples: []


**よくある間違い**

- `00b` を飛ばして `salmon_index` がない。
- `quant.sf` が一部のサンプルだけ存在する。
- `tx2gene.tsv` のtranscript IDと `quant.sf` の `Name` が合わず、集計後のgene数が少なすぎる。

**小さい練習**

`quant_status` で `False` のサンプルがあれば、そのサンプルのSalmonログを確認しよう。
